# Data Understanding & Time Audit

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

### Load the Workbook

In [2]:
file_path = "../data/raw/online_retail_II.xlsx"

df_0910 = pd.read_excel(file_path, sheet_name="Year 2009-2010")
df_1011 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

print("2009-2010 shape:", df_0910.shape)
print("2010-2011 shape:", df_1011.shape)

2009-2010 shape: (525461, 8)
2010-2011 shape: (541910, 8)


## Structural Consistency Check

In [3]:
df_0910.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
df_0910.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [6]:
df_1011.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [7]:
df_1011.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[ns]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [8]:
set(df_0910.columns) == set(df_1011.columns)

True

## Temporal Coverage Analysis

In [9]:
print("2009-2010 min:", df_0910["InvoiceDate"].min())
print("2009-2010 max:", df_0910["InvoiceDate"].max())

print("2010-2011 min:", df_1011["InvoiceDate"].min())
print("2010-2011 max:", df_1011["InvoiceDate"].max())

2009-2010 min: 2009-12-01 07:45:00
2009-2010 max: 2010-12-09 20:01:00
2010-2011 min: 2010-12-01 08:26:00
2010-2011 max: 2011-12-09 12:50:00


## Explicit Time Window Definition

To avoid temporal leakage, we define strict non-overlapping windows:

Training:
2009-12-01 ≤ date < 2010-12-01

Validation:
2010-12-01 ≤ date < 2011-12-01

In [11]:
train_df = df_0910[
    (df_0910["InvoiceDate"] >= "2009-12-01") &
    (df_0910["InvoiceDate"] < "2010-12-01")
]

validation_df = df_1011[
    (df_1011["InvoiceDate"] >= "2010-12-01") &
    (df_1011["InvoiceDate"] < "2011-12-01")
]

## Verification of Window Boundaries

In [12]:
print(train_df["InvoiceDate"].min(), train_df["InvoiceDate"].max())
print(validation_df["InvoiceDate"].min(), validation_df["InvoiceDate"].max())

2009-12-01 07:45:00 2010-11-30 19:35:00
2010-12-01 08:26:00 2011-11-30 17:42:00


## Save clean training and validation datasets

In [13]:
train_df.to_csv("../data/interim/train_window.csv", index=False)
validation_df.to_csv("../data/interim/validation_window.csv", index=False)

## Summary

- Confirmed structural consistency across sheets
- Identified boundary overlap in December 2010
- Enforced strict time-based splitting independent of sheet labels
- Generated clean, non-overlapping training and validation datasets

These datasets will be used for consistent RFM feature engineering in the next stage.
